In [1]:
from fpl_project.scripts import config
from fpl_project.scripts.data_utils.features_processing import sort_data
from fpl_project.scripts.model_utils.trainer import set_seed
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option('display.width', 1000)
pd.set_option('display.max_columns', 15)

In [2]:
set_seed(77)
data = sort_data(pd.read_parquet(config.FEATURES_ENGINEERING))
data.total_points = data.total_points.clip(lower=0)

In [ ]:
VARIANT = "mse"  # base-model variant from run_experiments: "mse" or "fplloss"
model_cols = ["glm", "xgboost", "mlp", "conv1d"]
key_cols = ["code", "season", "gw"]


def load_base_predictions(path):
    """Predykcje modeli bazowych z results/ z kolumnami wybranego wariantu przemianowanymi na model_cols."""
    df = pd.read_parquet(path)
    return df.rename(columns={f"{m}_{VARIANT}": m for m in model_cols}).reset_index(drop=True)


train = load_base_predictions(config.PREDS_VALID_PATH)  # season 25/26 - meta-model training set
test = load_base_predictions(config.PREDS_TEST_PATH)    # season 26/27, played gameweeks
predict_df = (
    load_base_predictions(config.PREDS_PREDICT_PATH)    # unplayed fixtures (is_played == False)
    if config.PREDS_PREDICT_PATH.exists()
    else pd.DataFrame()
)

y_train = train["y"]
y_test = test["y"]

print(f"Train (valid 25/26): {len(train)} | Test (26/27): {len(test)} | Nierozegrane: {len(predict_df)} | Wariant: {VARIANT}")

In [ ]:
X_train = train[model_cols].copy()
X_test = test[model_cols].copy()
preds_next_gw_models = predict_df[model_cols].copy() if not predict_df.empty else pd.DataFrame()

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")
print(f"Nulls in X_train: {X_train.isna().sum().sum()} | Nulls in X_test: {X_test.isna().sum().sum()}")

In [5]:
print("Korelacja predykcji modeli bazowych na zbiorze treningowym meta-modelu:")
print(X_train.corr().round(4))

Korelacja predykcji modeli bazowych na zbiorze treningowym meta-modelu:
            glm  xgboost     mlp  conv1d
glm      1.0000   0.9467  0.9231  0.9030
xgboost  0.9467   1.0000  0.9715  0.9469
mlp      0.9231   0.9715  1.0000  0.9504
conv1d   0.9030   0.9469  0.9504  1.0000


In [6]:
baseline_metrics = []
for col in model_cols:
    p_test_single = np.clip(X_test[col].values, 0, 15)
    baseline_metrics.append({
        "Model bazowy": col.upper(),
        "Test MSE": mean_squared_error(y_test, p_test_single),
        "Test MAE": mean_absolute_error(y_test, p_test_single),
        "Test R2": r2_score(y_test, p_test_single),
    })

baseline_df = pd.DataFrame(baseline_metrics).sort_values("Test MSE").reset_index(drop=True)
print("Wyniki pojedynczych modeli bazowych na zbiorze testowym:")
print(baseline_df.to_string(index=False))

Wyniki pojedynczych modeli bazowych na zbiorze testowym:
Model bazowy  Test MSE  Test MAE  Test R2
      CONV1D  4.398234  1.194727 0.301208
     XGBOOST  4.439729  1.191813 0.294615
         MLP  4.459688  1.191411 0.291444
         GLM  4.659779  1.230869 0.259653


In [7]:
# Non-negative linear regression (NNLS) on base-model predictions - best-performing meta-model
reg_meta = LinearRegression(positive=True, fit_intercept=True)
reg_meta.fit(X_train, y_train)

reg_importance = pd.DataFrame({
    "model_feature": X_train.columns,
    "coef": reg_meta.coef_,
}).sort_values(by="coef", ascending=False).reset_index(drop=True)

print(f"Intercept: {reg_meta.intercept_:.4f}")
print("Wagi modeli bazowych w Meta-Modelu Regresyjnym:")
print(reg_importance.to_string(index=False))

Intercept: -0.0899
Wagi modeli bazowych w Meta-Modelu Regresyjnym:
model_feature     coef
      xgboost 0.575594
       conv1d 0.407124
          glm 0.078373
          mlp 0.073026


In [ ]:
# Save meta-model predictions for season 26/27
meta_cols = ["name", "code", "team", "opponent_team", "position", "was_home", "season", "gw", "kickoff_time"]
kickoff = data[key_cols + ["kickoff_time"]]
test_meta = test.merge(kickoff, on=key_cols, how="left")[meta_cols].reset_index(drop=True)
test_meta["is_played"] = True

if not predict_df.empty and not preds_next_gw_models.empty:
    X_pred_meta = preds_next_gw_models[model_cols].copy().reset_index(drop=True)
    pred_meta = predict_df[meta_cols].copy().reset_index(drop=True)
    pred_meta["is_played"] = False
    season_2627_meta = pd.concat([test_meta, pred_meta], ignore_index=True)
else:
    X_pred_meta = None
    season_2627_meta = test_meta

out_df = season_2627_meta.copy()
p_test_reg = np.clip(reg_meta.predict(X_test), 0, 15)
if X_pred_meta is not None:
    p_next_reg = np.clip(reg_meta.predict(X_pred_meta), 0, 15)
    out_df["glm_meta"] = np.concatenate([p_test_reg, p_next_reg])
else:
    out_df["glm_meta"] = p_test_reg

out_df.to_parquet(config.PREDS_GLM_META, index=False)
print(f"Saved meta predictions: {out_df.shape} -> {config.PREDS_GLM_META}")

In [9]:
# Meta-model (NNLS) vs simple average
p_train_avg = np.clip(X_train.mean(axis=1).values, 0, 15)
p_test_avg = np.clip(X_test.mean(axis=1).values, 0, 15)

p_train_reg = np.clip(reg_meta.predict(X_train), 0, 15)
p_test_reg = np.clip(reg_meta.predict(X_test), 0, 15)

comparison_df = pd.DataFrame([
    {
        "Model": "Regression Meta-Model (NNLS)",
        "Train MSE": mean_squared_error(y_train, p_train_reg),
        "Train MAE": mean_absolute_error(y_train, p_train_reg),
        "Train R2": r2_score(y_train, p_train_reg),
        "Test MSE": mean_squared_error(y_test, p_test_reg),
        "Test MAE": mean_absolute_error(y_test, p_test_reg),
        "Test R2": r2_score(y_test, p_test_reg),
    },
    {
        "Model": "Simple Average (Zwykła Średnia)",
        "Train MSE": mean_squared_error(y_train, p_train_avg),
        "Train MAE": mean_absolute_error(y_train, p_train_avg),
        "Train R2": r2_score(y_train, p_train_avg),
        "Test MSE": mean_squared_error(y_test, p_test_avg),
        "Test MAE": mean_absolute_error(y_test, p_test_avg),
        "Test R2": r2_score(y_test, p_test_avg),
    },
]).sort_values("Test MSE").reset_index(drop=True)

print("Porównanie Meta-Modelu ze Zwykłą Średnią:")
print(comparison_df.to_string(index=False))

Porównanie Meta-Modelu ze Zwykłą Średnią:
                          Model  Train MSE  Train MAE  Train R2  Test MSE  Test MAE  Test R2
   Regression Meta-Model (NNLS)   3.649960   0.988145  0.338614  4.314172  1.191432 0.314564
Simple Average (Zwykła Średnia)   3.695871   0.997239  0.330295  4.377456  1.181731 0.304509
